# Section 1 — Evolution across Time

We explore how the popularity of first names in France has changed from 1900 to 2020,
using three complementary visualisations.

In [13]:
import altair as alt
import pandas as pd

alt.data_transformers.enable('json')

names = pd.read_csv("Names hints/dpt2020.csv", sep=";")
names = names[names.preusuel != '_PRENOMS_RARES']
names = names[names.dpt != 'XX']
names['annais'] = pd.to_numeric(names['annais'], errors='coerce')
names = names.dropna(subset=['annais'])
names['annais'] = names['annais'].astype(int)

names.head()

,sexe,preusuel,annais,dpt,nombre
10885,1,AADIL,1983,84,3
10886,1,AADIL,1992,92,3
10888,1,AAHIL,2016,95,3
10892,1,AARON,1962,75,3
10893,1,AARON,1976,75,3


## 1.a — Stacked proportion chart

Each band shows one name's share of all births in that year,
stacked on top of the others.
The combined height of all bands is the fraction of births covered by the top N names.
Click a name in the legend to highlight it.

In [14]:
N = 20
top = names.groupby('preusuel')['nombre'].sum().nlargest(N).index

yearly_total = names.groupby('annais')['nombre'].sum().reset_index(name='total')

annual = (names[names.preusuel.isin(top)]
          .groupby(['annais', 'preusuel'])['nombre'].sum().reset_index())
annual = annual.merge(yearly_total, on='annais')
annual['proportion'] = annual['nombre'] / annual['total']

sel = alt.selection_point(fields=['preusuel'], bind='legend')

alt.Chart(annual).mark_area(interpolate='monotone').encode(
    x=alt.X('annais:Q', axis=alt.Axis(format='d'), title='Year'),
    y=alt.Y('proportion:Q', stack='zero', axis=alt.Axis(format='%'), title='Share of births'),
    color=alt.Color('preusuel:N', title='Name'),
    opacity=alt.condition(sel, alt.value(1.0), alt.value(0.3)),
    tooltip=['preusuel:N', 'annais:Q', alt.Tooltip('proportion:Q', format='.2%')],
).add_params(sel).properties(width=750, height=400)

alt.Chart(...)

## 1.b — Ranked bar chart with year cursor

A horizontal bar chart of the most popular names for any given year.
Drag the slider to travel through time and watch the rankings shift.

In [15]:
N = 20
top = names.groupby('preusuel')['nombre'].sum().nlargest(N).index

annual_top = (names[names.preusuel.isin(top)]
              .groupby(['annais', 'preusuel'])['nombre'].sum().reset_index())

year_param = alt.param(
    name='year',
    bind=alt.binding_range(min=1900, max=2020, step=1, name='Year: '),
    value=2000,
)

alt.Chart(annual_top).mark_bar().encode(
    x=alt.X('nombre:Q', title='Count'),
    y=alt.Y('preusuel:N', sort='-x', title='Name'),
    color=alt.Color('nombre:Q', scale=alt.Scale(scheme='tealblues'), legend=None),
    tooltip=['preusuel:N', alt.Tooltip('nombre:Q', format=',')],
).transform_filter(
    alt.datum.annais == year_param
).add_params(year_param).properties(width=550, height=450)

alt.Chart(...)

## 1.c — Streamgraph

Each name is a trapezoidal ribbon whose width reflects its normalised share of births
for that decade. Linear interpolation between decade midpoints gives the straight-edged
trapezoid shape. The streams are centred so the chart reads as organic flow.
Click a name in the legend to highlight it.

In [16]:
N = 12
top = names.groupby('preusuel')['nombre'].sum().nlargest(N).index

sub = names[names.preusuel.isin(top)].copy()
sub['decade'] = (sub['annais'] // 10) * 10

by_decade = sub.groupby(['decade', 'preusuel'])['nombre'].sum().reset_index()
totals     = sub.groupby('decade')['nombre'].sum().reset_index(name='total')
by_decade  = by_decade.merge(totals, on='decade')
by_decade['proportion'] = by_decade['nombre'] / by_decade['total']

highlight = alt.selection_point(fields=['preusuel'], bind='legend')

alt.Chart(by_decade).mark_area(interpolate='linear').encode(
    x=alt.X('decade:Q', axis=alt.Axis(format='d'), title='Decade'),
    y=alt.Y('proportion:Q', stack='center', axis=None, title=''),
    color=alt.Color('preusuel:N', title='Name'),
    opacity=alt.condition(highlight, alt.value(0.85), alt.value(0.15)),
    tooltip=['preusuel:N', 'decade:Q',
             alt.Tooltip('proportion:Q', format='.2%')],
).add_params(highlight).properties(width=750, height=400)

alt.Chart(...)